# Medical QA Feature Engineering

Prototype shared features for MedQA and PubMedQA before promoting them into `src/coca_med/data/features.py`.

In [ ]:
from datasets import load_dataset

medqa = load_dataset("GBaker/MedQA-USMLE-4-options", split="train[:100]")
pubmedqa = load_dataset("qiaojin/PubMedQA", "pqa_labeled", split="train[:100]")

In [ ]:
def word_count(text):
    return len((text or "").split())

def medqa_features(row):
    options = row.get("options") or {}
    return {
        "question_words": word_count(row.get("question")),
        "num_choices": len(options),
        "num_metamap_phrases": len(row.get("metamap_phrases") or []),
        "has_meta_info": bool(row.get("meta_info")),
    }

def pubmedqa_features(row):
    context = row.get("context") or {}
    contexts = context.get("contexts", []) if isinstance(context, dict) else []
    meshes = context.get("meshes", []) if isinstance(context, dict) else []
    return {
        "question_words": word_count(row.get("question")),
        "context_words": sum(word_count(item) for item in contexts),
        "num_contexts": len(contexts),
        "num_mesh_terms": len(meshes),
        "has_long_answer": bool(row.get("long_answer")),
    }

medqa_features(medqa[0]), pubmedqa_features(pubmedqa[0])

In [ ]:
def difficulty_proxy(features):
    return (
        features.get("question_words", 0)
        + 0.25 * features.get("context_words", 0)
        + 2.0 * features.get("num_choices", 0)
    )

difficulty_proxy(medqa_features(medqa[0])), difficulty_proxy(pubmedqa_features(pubmedqa[0]))